# DHSVM Data Processing and S3 Integration

This notebook processes snow water equivalent (SWE) data from the DHSVM model for 4 basins in Washington State, uploads the results to AWS S3, and checks basin HUC12 overlaps with the training set. 

- Loading DHSVM output for each basin and aggregating daily SWE values.
- Merging basin data and uploading to S3 for downstream use.
- Identifying HUC12s in the DHSVM data regions which are missing in the training set (to include them)

In [ ]:
import io
import boto3
import pandas as pd
from itertools import chain
import geopandas as gpd

In [6]:
BRONZE_BUCKET = "snowml-bronze"
BRONZE_KEY    = "dhsvm/dhsvm_swe_all_basins.csv"
REGION        = "us-west-2"

BASE_URL = "https://data.cig.uw.edu/picea/mauger/misc/wrf_run_agg"

BASINS = {
    "Cedar":         f"{BASE_URL}/Aggregated.Values_Cedar",
    "Green":         f"{BASE_URL}/Aggregated.Values_Green",
    "Snohomish":     f"{BASE_URL}/Aggregated.Values_Snohomish",
    "Stilliguamish": f"{BASE_URL}/Aggregated.Values_Stilliguamish",
}

## Load DHSVM Output for Basins
For each basin, the notebook loads DHSVM output from remote URLs, processes daily mean SWE values and uploads them to the S3 bucket.

In [5]:
def load_dhsvm(basin_name, url):
    """Load DHSVM output for one basin, return daily mean SWE as a tidy DataFrame."""
    print(f"Loading {basin_name}...")
    df = pd.read_csv(url, sep=r'\s+')

    df.set_index(pd.DatetimeIndex(df.iloc[:, 0]), inplace=True)
    df.index.name = "day"

    df = df[["Swq"]].rename(columns={"Swq": "dhsvm_swe"})
    df = df.resample("D").mean()
    df["basin"] = basin_name
    df = df.reset_index()
    df["day"] = df["day"].dt.strftime("%Y-%m-%d")

    print(f"  {basin_name}: {len(df):,} rows, {df['day'].min()} → {df['day'].max()}")
    return df

In [7]:
dfs = []
for basin_name, url in BASINS.items():
    try:
        dfs.append(load_dhsvm(basin_name, url))
    except Exception as e:
        print(f"  [ERR] {basin_name}: {e}")

merged = pd.concat(dfs, ignore_index=True)
print(f"\nMerged shape: {merged.shape}")
print(merged.head())

Loading Cedar...
  Cedar: 14,611 rows, 1980-12-31 → 2020-12-31
Loading Green...
  Green: 14,611 rows, 1980-12-31 → 2020-12-31
Loading Snohomish...
  Snohomish: 12,783 rows, 1981-01-01 → 2015-12-31
Loading Stilliguamish...
  Stilliguamish: 12,784 rows, 1980-12-31 → 2015-12-31

Merged shape: (54789, 3)
          day     dhsvm_swe  basin
0  1980-12-31  0.000000e+00  Cedar
1  1981-01-01  0.000000e+00  Cedar
2  1981-01-02  0.000000e+00  Cedar
3  1981-01-03  9.138146e-11  Cedar
4  1981-01-04  0.000000e+00  Cedar


In [8]:
print("\nNull dhsvm_swe:", merged["dhsvm_swe"].isna().sum())
print("\nRows per basin:")
print(merged.groupby("basin").size())
print("\nDate range per basin:")
print(merged.groupby("basin")["day"].agg(["min", "max"]))


Null dhsvm_swe: 0

Rows per basin:
basin
Cedar            14611
Green            14611
Snohomish        12783
Stilliguamish    12784
dtype: int64

Date range per basin:
                      min         max
basin                                
Cedar          1980-12-31  2020-12-31
Green          1980-12-31  2020-12-31
Snohomish      1981-01-01  2015-12-31
Stilliguamish  1980-12-31  2015-12-31


In [9]:
s3 = boto3.client("s3", region_name=REGION)

buf = io.BytesIO()
merged.to_csv(buf, index=False)
buf.seek(0)

s3.put_object(Bucket=BRONZE_BUCKET, Key=BRONZE_KEY, Body=buf.getvalue())
print(f"\nUploaded → s3://{BRONZE_BUCKET}/{BRONZE_KEY}")


Uploaded → s3://snowml-bronze/dhsvm/dhsvm_swe_all_basins.csv


In [10]:
obj = s3.get_object(Bucket=BRONZE_BUCKET, Key=BRONZE_KEY)
df_check = pd.read_csv(io.BytesIO(obj["Body"].read()))
print(f"Verified: {df_check.shape} → s3://{BRONZE_BUCKET}/{BRONZE_KEY}")
print(df_check.head())

Verified: (54789, 3) → s3://snowml-bronze/dhsvm/dhsvm_swe_all_basins.csv
          day     dhsvm_swe  basin
0  1980-12-31  0.000000e+00  Cedar
1  1981-01-01  0.000000e+00  Cedar
2  1981-01-02  0.000000e+00  Cedar
3  1981-01-03  9.138146e-11  Cedar
4  1981-01-04  0.000000e+00  Cedar


## Check the Basin HUC12s that overlap with the HUC12s in the training set

In [ ]:
basins = {
    "Cedar":         "hucs_match_hsvm_cedar.geojson",
    "Green":         "hucs_match_hsvm_green.geojson",
    "Snohomish":     "hucs_match_hsvm_snoho.geojson",
    "Stilliguamish": "hucs_match_hsvm_stilly.geojson",
}

BASIN_HUCS = dict()
for basin, key in basins.items():
    gdf = gpd.read_file(f"s3://snowml-shape/{key}", storage_options={"anon": False})
    hucs = gdf["huc_id"].tolist()
    BASIN_HUCS[basin] = hucs
    

/home/sagemaker-user/.conda/envs/snowml/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option STORAGE_OPTIONS
  return ogr_read(
/home/sagemaker-user/.conda/envs/snowml/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option STORAGE_OPTIONS
  return ogr_read(
/home/sagemaker-user/.conda/envs/snowml/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option STORAGE_OPTIONS
  return ogr_read(
/home/sagemaker-user/.conda/envs/snowml/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option STORAGE_OPTIONS
  return ogr_read(


In [ ]:
MODEL_READY_BUCKET = "snowml-model-ready-stgnn"

def list_huc_files(bucket):
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket):
        for obj in page.get("Contents", []):
            k = obj["Key"]
            if k.startswith("model_ready_huc") and k.endswith(".csv") and obj["Size"] > 0:
                keys.append(k)
    return keys

huc_keys = list_huc_files(MODEL_READY_BUCKET)

huc_dfs_list = []
for key in huc_keys:
    huc_id = key.split("model_ready_huc")[1].split(".csv")[0]
    body = s3.get_object(Bucket=MODEL_READY_BUCKET, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(body))
    df["huc_id"] = huc_id
    huc_dfs_list.append(df)

huc_dfs = pd.concat(huc_dfs_list, ignore_index=True)
print(f"huc_dfs shape: {huc_dfs.shape}")
print(f"Unique HUCs: {huc_dfs['huc_id'].nunique()}")

huc_dfs shape: (8660352, 26)
Unique HUCs: 608


In [ ]:
training_hucs = set(huc_dfs['huc_id'].astype(str).unique())

print("Basin HUC12 overlap with training set:")
print("=" * 45)
for basin, hucs in BASIN_HUCS.items():
    in_training = [h for h in hucs if h in training_hucs]
    missing = [h for h in hucs if h not in training_hucs]
    print(f"\n{basin}: {len(in_training)}/{len(hucs)} in training set")
    if missing:
        print(f"  Not in training: {missing}")

total = list(chain.from_iterable(BASIN_HUCS.values()))
total_in = [h for h in total if h in training_hucs]
print(f"\nTotal: {len(total_in)}/{len(total)} basin HUCs in training set")

Basin HUC12 overlap with training set:

Cedar: 0/6 in training set
  Not in training: ['171100120101', '171100120102', '171100120103', '171100120104', '171100120105', '171100120107']

Green: 0/16 in training set
  Not in training: ['171100130101', '171100130102', '171100130103', '171100130104', '171100130105', '171100130106', '171100130201', '171100130202', '171100130203', '171100130204', '171100130205', '171100130301', '171100130302', '171100130303', '171100130304', '171100130305']

Snohomish: 21/51 in training set
  Not in training: ['171100100101', '171100100102', '171100100103', '171100100104', '171100100201', '171100100202', '171100100203', '171100100204', '171100100205', '171100100301', '171100100302', '171100100401', '171100100402', '171100100403', '171100100404', '171100100501', '171100100502', '171100100503', '171100100504', '171100100505', '171100100601', '171100100602', '171100100603', '171100110101', '171100110102', '171100110103', '171100110201', '171100110202', '171100110